In [1]:
import os
os.chdir('../../')

In [2]:
import os, torch
from tqdm import tqdm

def get_clip_score(pt_dir, clip, batch_size=64, device=None):
    dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
    files = sorted(p for p in (os.path.join(pt_dir, f) for f in os.listdir(pt_dir)) if p.endswith(".pt"))
    assert files, f"No .pt files in {pt_dir}"
    use_amp, outs = (dev == "cuda"), []

    with torch.no_grad():
        for i in tqdm(range(0, len(files), batch_size)):
            batch = [torch.load(p, map_location="cpu") for p in files[i:i+batch_size]]
            raws  = [ (d["raw"].unsqueeze(0) if d["raw"].ndim == 3 else d["raw"]) for d in batch ]
            conds = [ d["cond"] for d in batch ]
            raw   = torch.cat(raws, 0).to(dev, dtype=torch.bfloat16)

            with torch.autocast("cuda", torch.bfloat16, enabled=True):
                # reduction='none' → (B,)
                score = 1 - clip.get_cossim_loss(raw, conds, clamp_mode="hard", reduction="none")
            outs.append(score.detach().cpu())

    return float(torch.cat(outs).mean().item())


In [3]:
from utils.clip import CLIPEmbedder
device = 'cuda:0'
model_names = ['ViT-B/32', 'ViT-B/16', 'ViT-L/14', 'ViT-L/14@336px']

for model_name in model_names:
    clip = CLIPEmbedder(model_name=model_name, device=device)

    for step in [6, 5, 4, 3]:
        pt_dir = f'samplings/PixArt-Alpha/3.5/{step}/Euler/1000/euler_raw_0'
        try:
            score = get_clip_score(pt_dir, clip, batch_size=16, device=device)
            print(model_name, 'NFE :', step, 'Score :', score)
        except FileNotFoundError or AssertionError:
            continue
    print('======')

print('done')

100%|██████████| 63/63 [00:03<00:00, 16.21it/s]


ViT-B/32 NFE : 6 Score : 0.3057045042514801


100%|██████████| 63/63 [00:03<00:00, 18.00it/s]


ViT-B/32 NFE : 5 Score : 0.3025091290473938


100%|██████████| 63/63 [00:03<00:00, 17.83it/s]


...

ViT-L/14@336px NFE : 5 Score : 0.2612133324146271


100%|██████████| 63/63 [00:04<00:00, 13.00it/s]


ViT-L/14@336px NFE : 4 Score : 0.2553778290748596


100%|██████████| 63/63 [00:04<00:00, 12.87it/s]

ViT-L/14@336px NFE : 3 Score : 0.23466284573078156
done
